# Speaker voice-clone — QLoRA pilot (Unsloth)

Continued-pretraining-style **style** adaptation of a BASE model on his transcript corpus.
This is the **pilot** (validate the data shaping + training loop, not final quality).

**Recipe (from the `finetune_plan` project memory):**
- Model: **Llama-3.1-8B BASE** (NOT Instruct — Instruct RLHF sanitizes profanity, which we must keep, and injects an assistant register).
- Method: **QLoRA** (4-bit NF4 base + 16-bit LoRA) via Unsloth — 16-bit LoRA on 8B won't fit a T4/3060.
- LoRA: r=16, alpha=32, dropout=0.05, target all 7 linear projections.
- **FROZEN embed_tokens / lm_head** — the single biggest anti-forgetting lever (do NOT copy Unsloth's r=256+train-embeddings recipe; that's for teaching a NEW language).
- Objective: raw-text completion. Docs tokenized + EOS-joined + **pre-packed into 2048-token blocks** (all tokens used), NO chat template, NO loss masking.
- LR 1e-4, cosine + warmup 0.05, 2 epochs (cap 3), seq 2048, eff-batch 8 (bs2 x ga4), wd 0.01, adamw_8bit.
- Early-stop on **val** loss; train loss < 0.2 = overfit flag.

**EOS CONTRACT (the single most consequential shaping line — flagged by the formatter review):** the dataset formatter emits NO EOS on purpose (a literal EOS in `text` would be learned as emittable). This notebook appends `tokenizer.eos_token` between docs in the packed stream so video-docs get a real stop signal.

**Hardware:** Colab T4 16GB works; L4/A100 faster. Plan around T4.

## 1. Install

Uses the **official Unsloth Colab install** (continuously tested; `--no-deps` stops pip re-resolving the
stack into a conflict). This resolves a **modern TRL (>=0.22)**, so the trainer cell below uses
`max_length` (not `max_seq_length`) and `processing_class` (not `tokenizer`). If the latest stack ever
breaks, switch to the frozen fallback at the bottom of the cell AND apply the two renames noted there.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers peft trl triton
    !pip install --no-deps unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

# --- FROZEN FALLBACK (reproducible Nov-2024 stack; matches the TRL 0.12 API). If you use this,
#     in the trainer cell change `max_length=`->`max_seq_length=` and `processing_class=`->`tokenizer=`:
# %pip install -q "unsloth==2024.11.9" "unsloth_zoo==2024.11.7" "trl==0.12.2" "peft==0.13.2" \
#   "transformers==4.46.3" "datasets>=3.0,<4" "accelerate>=1.0,<1.2" "bitsandbytes>=0.44"

## 2. Environment check

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB")
else:
    raise SystemExit("No GPU — set Runtime > Change runtime type > GPU.")

## 3. Data

Get `train.jsonl` + `val.jsonl` (built locally by `python -m training.build_dataset`, under
`data/dataset/`) onto the runtime: in **VS Code, right-click each file → "Upload to Colab"**. Then run
the next cell — it *locates* them on the runtime. (We don't use `google.colab` `files.upload()` — that
browser widget hangs inside the VS Code Colab extension.) Each line is `{video_id, title, text}`; the
trainer reads only `text`.

In [ ]:
import os, glob

# Locate the uploaded files on the runtime (search the common landing spots, then glob as a fallback).
_SEARCH = [".", "/content", os.path.expanduser("~"), "/content/drive/MyDrive"]

def _find(name):
    for d in _SEARCH:
        p = os.path.join(d, name)
        if os.path.isfile(p):
            return p
    hits = glob.glob(f"/content/**/{name}", recursive=True) or glob.glob(f"**/{name}", recursive=True)
    return hits[0] if hits else None

TRAIN, VAL = _find("train.jsonl"), _find("val.jsonl")
assert TRAIN and VAL, (
    "train.jsonl / val.jsonl not found on the runtime. In VS Code, right-click each file under "
    "data/dataset/ and choose 'Upload to Colab' (they land in /content), then re-run this cell."
)
print("train:", TRAIN, os.path.getsize(TRAIN), "B | val:", VAL, os.path.getsize(VAL), "B")

# --- git-clone alternative (if you committed the files + have a PAT) ---
# import getpass; PAT = getpass.getpass("GitHub PAT: ")
# !git clone --depth 1 https://{PAT}@github.com/mehmettahacumurcu/youtuber-clone.git _repo
# TRAIN, VAL = "_repo/data/dataset/train.jsonl", "_repo/data/dataset/val.jsonl"

## 4. Load the 4-bit base model

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048  # pilot; raise to 4096 for the full run (still T4-feasible at bs1-2)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",  # BASE, pre-quantized 4-bit NF4 (ungated mirror)
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto (bf16 on Ampere+, fp16 on T4)
    load_in_4bit=True,
)
print("eos:", repr(tokenizer.eos_token), tokenizer.eos_token_id)
assert tokenizer.eos_token_id is not None, "tokenizer has no EOS — cannot honor the EOS contract"

## 5. Attach LoRA adapters (frozen embeddings)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    # All 7 linear projections. embed_tokens / lm_head are DELIBERATELY ABSENT -> frozen.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",  # ~30% less VRAM, longer context
    random_state=7,
    use_rslora=False,   # only worth it at r>=64 (full run)
    loftq_config=None,
)

# Verify the anti-forgetting lever actually held: embeddings + lm_head must be frozen.
frozen_ok = True
for name, param in model.named_parameters():
    if ("embed_tokens" in name or "lm_head" in name) and param.requires_grad:
        print("WARNING trainable embedding/head:", name)
        frozen_ok = False
assert frozen_ok, "embed_tokens/lm_head are trainable — they must be frozen for the pilot"
model.print_trainable_parameters()

## 6. Dataset — tokenize, EOS-join, pack into 2048-token blocks

We pre-pack manually: concatenate every doc's tokens with an EOS between docs, then split into fixed
`MAX_SEQ_LEN` blocks, so **every token is used**. (TRL's `packing=True` silently truncated each doc to a
single block on this stack — `Num examples = 42` ≈ #docs instead of ~240 — so we don't rely on it.)
The EOS between docs is the stop-signal contract; a block-count guard catches the truncation bug.

In [ ]:
from datasets import load_dataset, Dataset

EOS_ID = tokenizer.eos_token_id
BLOCK = MAX_SEQ_LEN

def pack(path):
    docs = load_dataset("json", data_files=path, split="train")
    ids = []
    for ex in docs:
        ids += tokenizer(ex["text"], add_special_tokens=False)["input_ids"] + [EOS_ID]  # EOS = doc boundary
    n = (len(ids) // BLOCK) * BLOCK                       # drop the ragged tail (< 1 block)
    blocks = [ids[i:i + BLOCK] for i in range(0, n, BLOCK)]
    ds = Dataset.from_dict({"input_ids": blocks, "labels": [b[:] for b in blocks]})
    return len(docs), len(ids), ds

n_tr, tok_tr, train_ds = pack(TRAIN)
n_va, tok_va, val_ds = pack(VAL)
print(f"train: {n_tr} docs -> {tok_tr:,} tokens -> {len(train_ds)} packed {BLOCK}-token blocks")
print(f"val:   {n_va} docs -> {tok_va:,} tokens -> {len(val_ds)} packed {BLOCK}-token blocks")

# Guard against the truncation bug (packing=True gave ~1 block/doc): we MUST get many blocks.
assert len(train_ds) > 3 * n_tr, f"only {len(train_ds)} blocks from {n_tr} docs — tokens still truncated"
assert any(EOS_ID in b for b in train_ds["input_ids"]), "EOS not embedded — stop-signal contract broken"
print("OK — all tokens packed, EOS doc-boundaries embedded.")

## 7. Trainer

Plain HF `Trainer` over the pre-packed blocks (labels = input_ids → causal-LM next-token loss; the
default collator just stacks the equal-length blocks). No SFT packing/text-field needed since the data is
already tokenized. Early-stop on **val** loss. `eval_strategy` is correct for transformers >=4.46.

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, default_data_collator

args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # eff-batch 8
    num_train_epochs=2,              # cap 3 if val loss still improving
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",           # older transformers: evaluation_strategy
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=7,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,   # blocks are equal-length input_ids+labels; just stack them
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## 8. Train

In [ ]:
train_out = trainer.train()
print(train_out)

## 9. Overfit / forgetting guards

360-600k tokens is small -> #1 risk is memorization/parroting + Turkish fluency loss.
Eyeball: (a) train loss not collapsing < 0.2, (b) val loss tracked down not up, (c) a generation that is
in-character, fluent Turkish, profanity intact, and NOT a verbatim training sentence.

In [ ]:
hist = trainer.state.log_history
train_losses = [h["loss"] for h in hist if "loss" in h]
eval_losses = [h["eval_loss"] for h in hist if "eval_loss" in h]
print("train loss:", [round(x, 3) for x in train_losses])
print("eval  loss:", [round(x, 3) for x in eval_losses])
if train_losses and train_losses[-1] < 0.2:
    print("OVERFIT FLAG: train loss < 0.2 — reduce epochs / LR, or scale the merged LoRA at inference.")
if len(eval_losses) >= 2 and eval_losses[-1] > min(eval_losses):
    print("VAL LOSS ROSE from its min — early-stop kept the best checkpoint (load_best_model_at_end).")

In [ ]:
FastLanguageModel.for_inference(model)

# Transcript-CONTINUATION prompt (base model — NOT an instruction). Tune format empirically per CLAUDE.md.
prompt = "Bugün sizlere anlatmak istediğim bir şey var. "
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.8, top_p=0.9,
                     repetition_penalty=1.1)
gen = tokenizer.decode(out[0], skip_special_tokens=True)
print(gen)

# Parroting probe: is the generation a verbatim chunk of training text?
import json
corpus = " ".join(json.loads(l)["text"] for l in open(TRAIN, encoding="utf-8"))
tail = gen[len(prompt):].strip()[:80]
print("\nPARROTING:", "verbatim-from-train!" if tail and tail in corpus else "not a verbatim match (good)")

## 10. Save

The **adapter** (~100-200 MB) is the pilot deliverable — small enough to push to HF. Merge to 16-bit
(for vLLM) or GGUF q4 (~5 GB, runs on the local RTX 3060 Ti) only when you actually need to serve.

In [ ]:
model.save_pretrained("speaker_lora")
tokenizer.save_pretrained("speaker_lora")
print("adapter saved to ./speaker_lora")

# --- persist beyond the ephemeral Colab runtime (pick one) ---
# HF private repo (adapter is small):
# from huggingface_hub import login; login()
# model.push_to_hub("<user>/speaker-llama31-8b-lora-pilot", private=True)
# tokenizer.push_to_hub("<user>/speaker-llama31-8b-lora-pilot", private=True)
#
# Google Drive copy:
# from google.colab import drive; drive.mount("/content/drive")
# !cp -r speaker_lora "/content/drive/MyDrive/speaker_lora"
#
# Merge for inference (later):
# model.save_pretrained_merged("speaker_merged_16bit", tokenizer, save_method="merged_16bit")
# model.save_pretrained_gguf("speaker_gguf", tokenizer, quantization_method="q4_k_m")

## 11. Get the model out of the runtime

The LoRA adapter (~100-200 MB) is the pilot deliverable. The Colab runtime is ephemeral, so pull it off.
Two ways that work in the **VS Code Colab extension** (the browser `files.download()` widget does NOT):
- zip it (next cell) and **right-click `speaker_lora.zip` in the VS Code explorer → Download**, or
- **push to a private HF repo** (uncomment in the next cell) — survives with no manual download.

To *generate* with it later, load the same base + this adapter (or use the merged / GGUF export from cell 10).

In [ ]:
# Zip the adapter, then download speaker_lora.zip via the VS Code explorer (right-click -> Download).
!zip -r -q speaker_lora.zip speaker_lora
import os
print("zipped:", os.path.abspath("speaker_lora.zip"), os.path.getsize("speaker_lora.zip"), "B")
print("VS Code: right-click 'speaker_lora.zip' in the explorer -> Download.")

# Or push the adapter straight to a private HF repo (survives the runtime, no manual download):
# from huggingface_hub import login; login()  # paste an HF token with write access
# model.push_to_hub("<user>/speaker-llama31-8b-lora-pilot", private=True)
# tokenizer.push_to_hub("<user>/speaker-llama31-8b-lora-pilot", private=True)

## Next: evaluation (the two mandatory arms)

Per `finetune_plan`, the pilot isn't done until:
1. **Blind A/B Llama-3.1-8B vs Gemma-2-9B base** on matched transcript-continuation prompts — let the
   *voice* decide (not MMLU). Re-run this notebook with `unsloth/gemma-2-9b-bnb-4bit`.
2. **No-fine-tune baseline: few-shot + RAG** — prove the fine-tune beats cheap prompting before committing.

Probes each checkpoint: Turkish fluency, profanity intact (must survive), and style-collapse / parroting.
Eval signal = held-out val loss **+** blind side-by-side vs his real text (don't over-trust val loss — it
carries ~0.18% boilerplate-catchphrase optimism).